In [ ]:
# 0. Imports
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, GlobalAveragePooling2D,
    Dense, Dropout, BatchNormalization, Input
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from sklearn.utils import class_weight
import matplotlib.pyplot as plt
import numpy as np
from skimage import exposure
from PIL import Image
import cv2
import os

# AdamW (TF >= 2.11 : tf.keras.optimizers.AdamW ; sinon tensorflow_addons)
try:
    from tensorflow.keras.optimizers import AdamW
except ImportError:
    from tensorflow.keras.optimizers.experimental import AdamW


In [ ]:
# 1. Configuration des chemins
train_dir = os.path.join('train')
test_dir = os.path.join('test')

In [ ]:

# 1.5 Prétraitement : CROP CENTRÉ + CLAHE
# ----
# Le crop centré (10%) supprime les marqueurs R/L et le texte machine
# qui faisaient dévier l'attention du modèle hors des poumons (effet Clever Hans).
# CLAHE améliore le contraste local des structures pulmonaires.
# Sortie : RGB 3 canaux (grayscale dupliqué) pour compatibilité DenseNet121.

CROP_RATIO = 0.10   # coupe 10% de chaque côté

def crop_center(img, ratio=CROP_RATIO):
    """Crop centré : supprime `ratio` des 4 bords."""
    h, w = img.shape[:2]
    ch, cw = int(h * ratio), int(w * ratio)
    return img[ch:h-ch, cw:w-cw]

def preprocess_xray(image):
    """
    1) clip + uint8
    2) crop centré (supprime marqueurs R/L)
    3) resize 224x224
    4) CLAHE sur le canal gris
    5) duplication 3 canaux + densenet preprocess_input
    """
    image = np.clip(image, 0, 255).astype(np.uint8)

    # Si l'image est RGB (ImageDataGenerator en color_mode='rgb'),
    # on passe en gris pour appliquer CLAHE puis on redupliquera
    if image.ndim == 3 and image.shape[2] == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    elif image.ndim == 3 and image.shape[2] == 1:
        gray = image[:, :, 0]
    else:
        gray = image

    # Crop centré
    gray = crop_center(gray, CROP_RATIO)

    # Resize à 224x224 après crop
    gray = cv2.resize(gray, (224, 224), interpolation=cv2.INTER_AREA)

    # CLAHE (contraste adaptatif)
    gray_f = exposure.equalize_adapthist(gray, clip_limit=0.02).astype(np.float32)
    gray_u8 = (gray_f * 255.0).astype(np.uint8)

    # Duplication 3 canaux puis preprocess DenseNet (ImageNet mean/std)
    rgb = np.stack([gray_u8, gray_u8, gray_u8], axis=-1).astype(np.float32)
    rgb = densenet_preprocess(rgb)
    return rgb

In [ ]:
def count_classes(directory):
    classes = ['NORMAL', 'PNEUMONIA']
    counts = []
    for cls in classes:
        path = os.path.join(directory, cls)
        counts.append(len(os.listdir(path)))
    return counts

# On recupere les comptes
train_counts = count_classes(train_dir)
test_counts = count_classes(test_dir)

# Visualisation
labels = ['Normal', 'Pneumonie']
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Graphique Train
ax[0].bar(labels, train_counts, color=['skyblue', 'salmon'])
ax[0].set_title(f'Entrainement (Total: {sum(train_counts)})')

# Graphique Test
ax[1].bar(labels, test_counts, color=['skyblue', 'salmon'])
ax[1].set_title(f'Test/Val (Total: {sum(test_counts)})')

plt.show()

print(f"Ratio Train: {train_counts[1]/train_counts[0]:.2f}x plus de pneumonies")

In [ ]:

# 3. Flux de données (SANS GAN — on entraîne sur le train original)
# ----
# - target_size=(224,224) mais le crop 10% est appliqué par preprocess_xray
# - color_mode='rgb' pour nourrir DenseNet121 (3 canaux)
# - augmentation classique (rotation, flip, zoom) pour compenser l'imbalance

BATCH = 32

train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.08,
    height_shift_range=0.08,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15],   # <-- color jitter (cf. CheXNet enhanced 2025)
    preprocessing_function=preprocess_xray,
    fill_mode='nearest'
)

train_generator = train_datagen.flow_from_directory(
    train_dir,                       # <-- dataset original, pas de GAN
    target_size=(224, 224),
    batch_size=BATCH,
    class_mode='categorical',
    color_mode='rgb'
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_xray)
validation_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=BATCH,
    class_mode='categorical',
    color_mode='rgb',
    shuffle=False
)

print("Classes :", train_generator.class_indices)

In [ ]:

# 5. Modèle : DenseNet121 pré-entraîné (ImageNet) + tête custom
# ----
# Architecture de référence en imagerie médicale (base de CheXNet, Stanford).
# On part des poids ImageNet puis on entraîne en 2 phases.

def build_densenet_model(num_classes=2):
    base = DenseNet121(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base.trainable = False   # Phase 1 : backbone gelé

    inputs = Input(shape=(224, 224, 3))
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs, name='DenseNet121_Pneumonie')
    return model, base

model, base_model = build_densenet_model(num_classes=2)
print(f"Paramètres totaux    : {model.count_params():,}")
print(f"Paramètres entraînables (phase 1) : {sum([tf.size(v).numpy() for v in model.trainable_weights]):,}")

In [ ]:

# 5.5 Focal Loss (catégorielle)
# ----
# Mieux que categorical_crossentropy + class_weight pour un dataset
# déséquilibré (ratio 2.7:1). Donne plus de poids aux exemples difficiles.

def categorical_focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.math.pow(1.0 - y_pred, gamma)
        return tf.reduce_sum(weight * ce, axis=-1)
    return loss

focal = categorical_focal_loss(gamma=2.0, alpha=0.25)

In [ ]:

# 6. Callbacks + Entraînement (ou CHARGEMENT d'un modèle existant)
# ----
# LOAD_EXISTING_MODEL = True  -> on charge best_model_densenet.keras et on saute l'entraînement
# LOAD_EXISTING_MODEL = False -> on relance l'entraînement en 2 phases (1-2h)

LOAD_EXISTING_MODEL = True          # <-- mets False si tu veux ré-entraîner
MODEL_PATH = 'best_model_densenet_balanced.keras'

if LOAD_EXISTING_MODEL and os.path.exists(MODEL_PATH):
    print(f"Chargement du modele existant : {MODEL_PATH}")
    # compile=False car focal est une fonction custom, on recompile apres
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    model.compile(
        optimizer=AdamW(learning_rate=1e-5, weight_decay=1e-4),
        loss=focal,
        metrics=['accuracy']
    )
    # Historique vide (on n affichera pas les courbes d apprentissage)
    history = type('H', (), {})()
    history.history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
    print("Modele charge. Entrainement saute.")
    print(f"   Parametres totaux : {model.count_params():,}")

else:
    print("Entrainement complet (2 phases)...")

    early_stop = EarlyStopping(monitor='val_loss', patience=6,
                               restore_best_weights=True, verbose=1)
    reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                   patience=3, min_lr=1e-7, verbose=1)
    checkpoint = ModelCheckpoint(MODEL_PATH,
                                 monitor='val_loss', save_best_only=True, verbose=1)

    # --- Poids de classes ---
    labels  = train_generator.classes
    weights = class_weight.compute_class_weight(
        class_weight='balanced', classes=np.unique(labels), y=labels)
    dict_weights = {0: weights[0], 1: weights[1]}
    print(f"Poids classes : {dict_weights}")

    # --- PHASE 1 ---
    print("\n" + "="*60)
    print("PHASE 1 - Entrainement de la tete (backbone gele)")
    print("="*60)
    model.compile(
        optimizer=AdamW(learning_rate=1e-3, weight_decay=1e-4),
        loss=focal, metrics=['accuracy']
    )
    history_1 = model.fit(
        train_generator, epochs=10,
        validation_data=validation_generator,
        class_weight=dict_weights,
        callbacks=[early_stop, reduce_lr, checkpoint]
    )

    # --- PHASE 2 ---
    print("\n" + "="*60)
    print("PHASE 2 - Fine-tuning des derniers blocs")
    print("="*60)
    base_model.trainable = True
    for layer in base_model.layers[:-40]:
        layer.trainable = False
    for layer in base_model.layers:
        if isinstance(layer, BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=AdamW(learning_rate=1e-5, weight_decay=1e-4),
        loss=focal, metrics=['accuracy']
    )
    history_2 = model.fit(
        train_generator, epochs=20,
        validation_data=validation_generator,
        class_weight=dict_weights,
        callbacks=[early_stop, reduce_lr, checkpoint]
    )

    # Fusion des historiques
    history = type('H', (), {})()
    history.history = {
        k: history_1.history[k] + history_2.history[k]
        for k in history_1.history.keys()
    }

In [ ]:

# 7. Fine-tuning équilibré (warm start + oversampling + focal pondéré)
# ----
# DO_BALANCED_FINETUNE = True  -> exécute le warm start (~20-40 min)
# DO_BALANCED_FINETUNE = False -> charge best_model_densenet_balanced.keras s'il existe, sinon garde le modèle actuel

import shutil

DO_BALANCED_FINETUNE = False            # <-- passe à False après la première exécution réussie
BALANCED_MODEL_PATH  = 'best_model_densenet_balanced.keras'
BALANCED_DIR         = 'train_balanced'
OVERSAMPLE_FACTOR    = 3               # NORMAL dupliqué x3 → ~4023 vs 3875 PNEUMONIA

if DO_BALANCED_FINETUNE:

    # ---------- 7.1 Construction du dataset rééquilibré (symlinks) ----------
    src_normal    = os.path.join(train_dir, 'NORMAL')
    src_pneumonia = os.path.join(train_dir, 'PNEUMONIA')
    dst_normal    = os.path.join(BALANCED_DIR, 'NORMAL')
    dst_pneumonia = os.path.join(BALANCED_DIR, 'PNEUMONIA')

    if os.path.exists(BALANCED_DIR):
        shutil.rmtree(BALANCED_DIR)
    os.makedirs(dst_normal)
    os.makedirs(dst_pneumonia)

    def link_or_copy(src, dst):
        try:
            os.symlink(os.path.abspath(src), dst)
        except OSError:
            shutil.copy(src, dst)

    # PNEUMONIA : 1x (original)
    for f in os.listdir(src_pneumonia):
        if f.startswith('.'):
            continue
        link_or_copy(os.path.join(src_pneumonia, f), os.path.join(dst_pneumonia, f))

    # NORMAL : x OVERSAMPLE_FACTOR (noms différents pour éviter collision)
    for dup in range(OVERSAMPLE_FACTOR):
        for f in os.listdir(src_normal):
            if f.startswith('.'):
                continue
            dst_name = f if dup == 0 else f'dup{dup}_{f}'
            link_or_copy(os.path.join(src_normal, f), os.path.join(dst_normal, dst_name))

    n_norm = len([x for x in os.listdir(dst_normal) if not x.startswith('.')])
    n_pneu = len([x for x in os.listdir(dst_pneumonia) if not x.startswith('.')])
    print(f"Dataset rééquilibré : {n_norm} NORMAL | {n_pneu} PNEUMONIA | ratio {n_pneu/n_norm:.2f}:1")

    # ---------- 7.2 Nouveau générateur sur le dataset équilibré ----------
    train_generator_balanced = train_datagen.flow_from_directory(
        BALANCED_DIR,
        target_size=(224, 224),
        batch_size=BATCH,
        class_mode='categorical',
        color_mode='rgb'
    )

    # ---------- 7.3 Focal loss pondéré (alpha plus élevé sur NORMAL) ----------
    def categorical_focal_loss_weighted(gamma=2.0, alpha=(0.7, 0.3)):
        alpha_t = tf.constant(alpha, dtype=tf.float32)
        def loss(y_true, y_pred):
            y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
            ce = -y_true * tf.math.log(y_pred)
            weight = alpha_t * tf.math.pow(1.0 - y_pred, gamma)
            return tf.reduce_sum(weight * ce, axis=-1)
        return loss

    focal_weighted = categorical_focal_loss_weighted(gamma=2.0, alpha=(0.7, 0.3))

    # ---------- 7.4 Recompile en warm start (LR très bas) ----------
    model.compile(
        optimizer=AdamW(learning_rate=5e-6, weight_decay=1e-4),
        loss=focal_weighted,
        metrics=['accuracy']
    )

    # ---------- 7.5 Callbacks ----------
    checkpoint_bal = ModelCheckpoint(
        BALANCED_MODEL_PATH, monitor='val_loss',
        save_best_only=True, verbose=1
    )
    early_bal = EarlyStopping(
        monitor='val_loss', patience=4,
        restore_best_weights=True, verbose=1
    )
    reduce_bal = ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=2, min_lr=1e-8, verbose=1
    )

    # ---------- 7.6 Fine-tuning court (warm start) ----------
    print("\n" + "="*60)
    print("WARM START - Fine-tuning équilibré (~8 epochs, 20-40 min)")
    print("="*60)
    history_balanced = model.fit(
        train_generator_balanced,
        epochs=8,
        validation_data=validation_generator,
        callbacks=[checkpoint_bal, early_bal, reduce_bal]
    )

    print(f"\nModele equilibre sauvegarde : {BALANCED_MODEL_PATH}")

elif os.path.exists(BALANCED_MODEL_PATH):
    print(f"Chargement du modele equilibre existant : {BALANCED_MODEL_PATH}")
    model = tf.keras.models.load_model(BALANCED_MODEL_PATH, compile=False)
    model.compile(
        optimizer=AdamW(learning_rate=1e-5, weight_decay=1e-4),
        loss=focal,
        metrics=['accuracy']
    )
    print("Modele equilibre charge. La suite du notebook utilisera ce modele.")

else:
    print("Aucun modele equilibre disponible. Le notebook continuera avec le modele initial.")

In [ ]:
model.summary()

In [ ]:
# 8. Evaluation du modele sur le jeu de test
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Creer un generateur d'evaluation SANS shuffle
eval_datagen = ImageDataGenerator(preprocessing_function=preprocess_xray)
eval_generator = eval_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical',
    color_mode='rgb',
    shuffle=False
)

# Predictions (softmax retourne 2 probabilites)
y_pred_prob = model.predict(eval_generator)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = eval_generator.classes

# Noms des classes
class_names = list(eval_generator.class_indices.keys())
print("Classes:", eval_generator.class_indices)
print("\n" + "="*50)
print("RAPPORT DE CLASSIFICATION")
print("="*50)
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:

# 9. Matrice de confusion
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Prediction')
plt.ylabel('Realite')
plt.title('Matrice de Confusion')
plt.tight_layout()
plt.show()

print(f"\nAccuracy globale: {np.sum(np.diag(cm)) / np.sum(cm) * 100:.2f}%")

In [ ]:

# 10bis. Threshold Tuning — trouver le seuil optimal
# ----
# On ajuste le seuil de decision sur la classe PNEUMONIA
# pour maximiser le F1 macro (equilibre NORMAL vs PNEUMONIA).
# Aucun reentrainement, juste post-processing des probabilites.

from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    confusion_matrix, classification_report
)

# On utilise y_pred_prob et y_true deja calcules par la Cell 10 (evaluation)
proba_pneumo = y_pred_prob[:, 1]   # probabilite de la classe PNEUMONIA

# --- Balayage de seuils ---
thresholds = np.arange(0.30, 0.90, 0.01)
rows = []
for t in thresholds:
    y_pred_t = (proba_pneumo > t).astype(int)
    rows.append({
        'threshold': t,
        'f1_normal': f1_score(y_true, y_pred_t, pos_label=0, zero_division=0),
        'f1_pneumo': f1_score(y_true, y_pred_t, pos_label=1, zero_division=0),
        'f1_macro':  f1_score(y_true, y_pred_t, average='macro', zero_division=0),
        'recall_normal': recall_score(y_true, y_pred_t, pos_label=0, zero_division=0),
        'recall_pneumo': recall_score(y_true, y_pred_t, pos_label=1, zero_division=0),
        'precision_normal': precision_score(y_true, y_pred_t, pos_label=0, zero_division=0),
        'precision_pneumo': precision_score(y_true, y_pred_t, pos_label=1, zero_division=0),
    })

# --- Trouver le seuil optimal (max F1 macro) ---
best = max(rows, key=lambda r: r['f1_macro'])
best_t = best['threshold']
print(f"Seuil optimal (F1 macro) : {best_t:.2f}")
print(f"  F1 macro    : {best['f1_macro']:.4f}")
print(f"  Recall NORMAL   : {best['recall_normal']:.4f}")
print(f"  Recall PNEUMO   : {best['recall_pneumo']:.4f}")
print(f"  Precision NORMAL: {best['precision_normal']:.4f}")
print(f"  Precision PNEUMO: {best['precision_pneumo']:.4f}")

# --- Comparaison : seuil 0.5 (defaut) vs seuil optimal ---
row_05 = [r for r in rows if abs(r['threshold'] - 0.50) < 0.005][0]
print("\n" + "="*60)
print(f"{'Metrique':<20} {'Seuil 0.50':>12} {'Seuil '+f'{best_t:.2f}':>12} {'Delta':>10}")
print("="*60)
for key, label in [('f1_macro', 'F1 macro'),
                   ('recall_normal', 'Recall NORMAL'),
                   ('recall_pneumo', 'Recall PNEUMO'),
                   ('precision_normal', 'Prec. NORMAL'),
                   ('precision_pneumo', 'Prec. PNEUMO')]:
    v05 = row_05[key]
    vbest = best[key]
    delta = vbest - v05
    sign = '+' if delta >= 0 else ''
    print(f"{label:<20} {v05:>12.4f} {vbest:>12.4f} {sign}{delta:>9.4f}")
print("="*60)

# --- Graphique : evolution des metriques vs seuil ---
fig, ax = plt.subplots(figsize=(12, 6))
ts = [r['threshold'] for r in rows]
ax.plot(ts, [r['recall_normal'] for r in rows], label='Recall NORMAL', color='#3498db', lw=2)
ax.plot(ts, [r['recall_pneumo'] for r in rows], label='Recall PNEUMONIA', color='#e74c3c', lw=2)
ax.plot(ts, [r['f1_macro']      for r in rows], label='F1 macro', color='#2ecc71', lw=2.5, linestyle='--')
ax.axvline(x=0.50, color='gray', linestyle=':', alpha=0.7, label='Seuil defaut (0.50)')
ax.axvline(x=best_t, color='black', linestyle=':', alpha=0.9, label=f'Seuil optimal ({best_t:.2f})')
ax.set_xlabel('Seuil de decision (proba PNEUMONIA)')
ax.set_ylabel('Score')
ax.set_title('Evolution des metriques selon le seuil')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Matrice de confusion au seuil optimal ---
y_pred_best = (proba_pneumo > best_t).astype(int)
cm_best = confusion_matrix(y_true, y_pred_best)
class_names_cm = ['NORMAL', 'PNEUMONIA']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, cm_, title in [(axes[0], cm, f'Seuil 0.50 (defaut)'),
                        (axes[1], cm_best, f'Seuil optimal ({best_t:.2f})')]:
    sns.heatmap(cm_, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names_cm, yticklabels=class_names_cm, ax=ax)
    ax.set_xlabel('Prediction')
    ax.set_ylabel('Realite')
    ax.set_title(title)
plt.tight_layout()
plt.show()

# --- Rapport final ---
print("\nRAPPORT DE CLASSIFICATION AU SEUIL OPTIMAL")
print("="*60)
print(classification_report(y_true, y_pred_best, target_names=class_names_cm))

# --- Stocke le seuil pour reutilisation (ex: frontend) ---
OPTIMAL_THRESHOLD = best_t
print(f"\nSeuil a utiliser en production : OPTIMAL_THRESHOLD = {OPTIMAL_THRESHOLD:.2f}")

In [ ]:

# 10. Courbes d'entrainement (seulement si on a entraine)

if len(history.history.get('loss', [])) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history.history['loss'], label='Train Loss')
    ax1.plot(history.history['val_loss'], label='Val Loss')
    ax1.set_title('Evolution de la Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(True)

    ax2.plot(history.history['accuracy'], label='Train Accuracy')
    ax2.plot(history.history['val_accuracy'], label='Val Accuracy')
    ax2.set_title("Evolution de l'Accuracy")
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.grid(True)

    plt.tight_layout()
    plt.show()
else:
    print("Modele charge depuis le disque - pas de courbes d entrainement a afficher.")